In [3]:
# =============================================================================
# Tri-Modal Time Aligner — UnionGrid Lite v3 (cleaned schema)
# -----------------------------------------------------------------------------
# Goal:
#   Align EEG / EMG / Eye Tracking to a common 250 Hz grid (union span by default)
#   using your existing Timestamp_seconds (from the cleaner).

# Outputs:
#   <stem>_synchronized_corrected.csv
#   <stem>_synchronized_corrected.json
#   synchronization_summary.csv
# =============================================================================



from __future__ import annotations

import os, glob, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal
from scipy.interpolate import interp1d

warnings.filterwarnings("ignore")

# -------------------------
# CONFIG
# -------------------------
# NOTE (Jupyter-friendly):
# If your notebook loop sets IN_DIR / OUT_DIR before calling run_sync_batch(),
# these defaults will NOT overwrite them.
if "IN_DIR" not in globals():
    IN_DIR  = r"/home/tsultan1/IROS/Data/Sub-1/cleaned"  # default fallback only
if "OUT_DIR" not in globals():
    OUT_DIR = r"/home/tsultan1/IROS/Data/Sub-1/cleaned/synchronized_proper_lite_union_v3"  # default fallback only

GRID_STYLE   = "fixed"     # "fixed" (target grid) or "native"
TARGET_HZ    = 250
SPAN_STYLE   = "all"       # "all" (union) or "overlap"
GRID_STEP_MS = 1000.0 / TARGET_HZ

# Lag estimation resolution
XCORR_HZ        = 1000
XCORR_WIN_SEC   = 12.0
N_BINS          = 5
N_SELECT        = 3
MIN_ACCEPT      = 2

# Filter settings for lag estimation
EEG_BAND     = (1.0, 40.0)
EMG_ENV_LP   = 10.0
ET_VEL_BAND  = (0.5, 10.0)
FILT_ORDER   = 3

# Anti-aliasing (before downsampling to TARGET_HZ)
AA_ON       = True
AA_ORDER    = 8
AA_MARGIN   = 0.45
AA_TOL      = 1.05
DISCRETE_COLS = {
    "ET_ValidityLeftEye","ET_ValidityRightEye","ET_Blink","ET_Fixation","ET_Worn"
}

# Guards / fallbacks
EMG_EEG_PEAKR_MIN       = 4.0
EMG_EEG_ABS_REJECT_MS   = 150.0
FALLBACK_CLIP_MS        = 50.0

ET_PEAKR_MIN            = 8.0
ET_MAX_ABS_MS           = 60.0
ET_REVERT_MARGIN        = 0.007

# Robust bounds (hard limits)
EMG_EEG_SEARCH_MAX_MS   = 200.0
EDGE_HIT_FRAC           = 0.90
AGREE_TOL_MS            = 30.0
TAPER_FOR_CORR          = True
FILL_NAN_FOR_CORR       = True

# Optional soft prior (tune for your rig)
EMG_EEG_PRIOR_MS = -28.0
PRIOR_STRENGTH   = 0.6

# Output formatting
ROUND_DECIMALS     = 4
WRITE_JSON         = True
KEEP_META_COLUMNS  = True  # keep subject_id/task/trial as constants if present


# =========================
# Small math helpers
# =========================
def _safe_float(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return default


def _butter_bandpass(x, fs, f1, f2, order=FILT_ORDER):
    try:
        ny = fs / 2.0
        if 0 < f1 < f2 < ny:
            b, a = signal.butter(order, [f1/ny, f2/ny], btype="band")
            return signal.filtfilt(b, a, x)
    except Exception:
        pass
    return x


def _butter_lowpass(x, fs, fc, order=FILT_ORDER):
    try:
        ny = fs / 2.0
        if 0 < fc < ny:
            b, a = signal.butter(order, fc/ny, btype="low")
            return signal.filtfilt(b, a, x)
    except Exception:
        pass
    return x


def _interp_series(t_src, y_src, t_dst, kind="linear"):
    m = np.isfinite(t_src) & np.isfinite(y_src)
    if m.sum() < 2:
        return np.full_like(t_dst, np.nan, dtype=float)
    f = interp1d(t_src[m], y_src[m], kind=kind, bounds_error=False, fill_value=np.nan)
    return f(t_dst)


def _grid_round_ms(ms: float) -> float:
    return float(np.round(ms / GRID_STEP_MS) * GRID_STEP_MS)


def _overlap_span(t1, t2):
    return max(np.nanmin(t1), np.nanmin(t2)), min(np.nanmax(t1), np.nanmax(t2))


def _infer_fs(ts: np.ndarray) -> float:
    if ts is None or len(ts) < 3:
        return 0.0
    dt = np.diff(ts)
    dt = dt[(dt > 0) & (dt < 1.0)]
    if len(dt) == 0:
        return 0.0
    return 1.0 / np.median(dt)


def _pick_windows_by_energy(t, x, a, b, n_bins=N_BINS, n_select=N_SELECT, win_sec=XCORR_WIN_SEC):
    if b - a <= win_sec + 1.0:
        return [(max(a, a + 0.5), min(b, b - 0.5))]

    edges = np.linspace(a + 0.5, b - 0.5, n_bins + 1)
    cand = []
    for i in range(n_bins):
        start = edges[i]
        end = min(start + win_sec, edges[i+1])
        m = (t >= start) & (t <= end)
        if m.sum() < 10:
            continue
        cand.append((np.nanvar(x[m]), start, end))

    cand.sort(key=lambda z: z[0], reverse=True)
    wins = [(s, e) for _, s, e in cand[:max(1, n_select)]]
    return wins if wins else [(a + 0.5, min(b - 0.5, a + 0.5 + win_sec))]


def _nan_linear_fill(y):
    y = np.asarray(y, dtype=float)
    m = np.isfinite(y)
    if m.sum() < 2:
        return np.nan_to_num(y)
    idx = np.arange(len(y))
    f = interp1d(idx[m], y[m], bounds_error=False, fill_value=(y[m][0], y[m][-1]))
    out = y.copy()
    out[~m] = f(idx[~m])
    return out


def _aa_pre_filter(y, fs_native, fs_target):
    """Zero-phase lowpass before resampling (only if downsampling)."""
    if not np.isfinite(fs_native) or fs_native <= 0:
        return y
    fc = min(0.49 * fs_target, AA_MARGIN * fs_target)
    ny = fs_native / 2.0
    if fc >= ny:
        return y
    y2 = _nan_linear_fill(y)
    try:
        b, a = signal.butter(AA_ORDER, fc / ny, btype="low")
        padlen = max(0, min(3 * max(len(a), len(b)), len(y2) - 1))
        if padlen < 1:
            return y2
        return signal.filtfilt(b, a, y2, padlen=padlen)
    except Exception:
        return y


# =========================
# IO + stream partitioning
# =========================
def ingest_clean_csv(csv_path: str | Path) -> pd.DataFrame:
    """Read and enforce Timestamp_seconds monotonicity (uses existing Timestamp_seconds)."""
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    if "Timestamp_seconds" not in df.columns:
        raise ValueError("Timestamp_seconds not found. Your cleaner should have created it.")
    df["Timestamp_seconds"] = pd.to_numeric(df["Timestamp_seconds"], errors="coerce")
    df = df.sort_values("Timestamp_seconds").drop_duplicates("Timestamp_seconds")
    return df


def partition_modal_streams(df: pd.DataFrame):
    """
    Critical fix:
      Build per-modality substreams by keeping rows where that modality has ANY data.
      This prevents fs/lag being computed over long NaN stretches.
    """
    emg_cols = [c for c in ["Ch1 EMG raw","Ch2 EMG raw","Ch3 EMG raw","Ch4 EMG raw"] if c in df.columns]
    eeg_cols = [c for c in [f"Ch{i}" for i in range(1,9)] if c in df.columns]
    et_cols  = [c for c in df.columns if c.startswith("ET_")]

    def _make_stream(cols):
        if not cols:
            return df.iloc[0:0].copy()
        sub = df[["Timestamp_seconds"] + cols].copy()
        sub = sub.dropna(subset=cols, how="all")
        sub = sub.sort_values("Timestamp_seconds").drop_duplicates("Timestamp_seconds")
        return sub

    emg = _make_stream(emg_cols)
    eeg = _make_stream(eeg_cols)
    et  = _make_stream(et_cols)

    return emg, eeg, et, emg_cols, eeg_cols, et_cols


def extract_constant_meta(df: pd.DataFrame) -> dict:
    """Preserve constant meta columns if they exist."""
    meta = {}
    for k in ["subject_id", "task", "trial"]:
        if k in df.columns:
            s = pd.to_numeric(df[k], errors="coerce")
            v = s.dropna().iloc[0] if s.dropna().shape[0] else np.nan
            meta[k] = int(v) if np.isfinite(v) else np.nan
    return meta


# =========================
# Lag estimation
# =========================
def _prep_corr_window(t, x, fs, a, b, bp=None):
    m = (t >= a) & (t <= b)
    if m.sum() < 10:
        return None, None
    n = int((b - a) * fs)
    if n < 100:
        return None, None

    th = np.linspace(a, b, n, endpoint=False)
    xh = _interp_series(t[m], x[m], th)

    if FILL_NAN_FOR_CORR:
        xh = _nan_linear_fill(xh)

    xh = (xh - np.nanmean(xh)) / (np.nanstd(xh) + 1e-12)
    xh = np.nan_to_num(xh)

    if bp is not None:
        xh = _butter_bandpass(xh, fs, bp[0], bp[1])

    if TAPER_FOR_CORR:
        w = np.hanning(len(xh))
        if w.max() > 0:
            xh = xh * w

    return th, xh


def _bounded_peak(lags, corr, fs, max_ms):
    max_samp = int(np.floor((max_ms/1000.0) * fs))
    m = (lags >= -max_samp) & (lags <= max_samp)
    if not np.any(m):
        return np.nan, np.nan, np.nan

    l = lags[m]
    c = corr[m]
    i = int(np.argmax(np.abs(c)))
    peak = c[i]
    pr = float(np.abs(peak) / (np.median(np.abs(c)) + 1e-12))
    return float(l[i]), float(peak), pr


def lag_xcorr_ms(x1, x2, fs, max_ms):
    corr = signal.correlate(x1, x2, mode="full")
    lags = signal.correlation_lags(len(x1), len(x2), mode="full")
    lag_s, _, pr = _bounded_peak(lags, corr, fs, max_ms)
    if not np.isfinite(lag_s):
        return np.nan, np.nan
    return (lag_s / fs) * 1000.0, pr


def lag_gccphat_ms(x1, x2, fs, max_ms):
    n = int(2 ** np.ceil(np.log2(len(x1) + len(x2) - 1)))
    X1 = np.fft.rfft(x1, n=n)
    X2 = np.fft.rfft(x2, n=n)
    R = X1 * np.conj(X2)
    R = R / (np.abs(R) + 1e-12)
    corr = np.fft.irfft(R, n=n)

    corr = np.concatenate((corr[-(len(x1)-1):], corr[:len(x2)]))
    lags = np.arange(-len(x1)+1, len(x2))

    lag_s, _, pr = _bounded_peak(lags, corr, fs, max_ms)
    if not np.isfinite(lag_s):
        return np.nan, np.nan
    return (lag_s / fs) * 1000.0, pr


def estimate_lag_emg_to_eeg(emg: pd.DataFrame, eeg: pd.DataFrame, emg_cols, eeg_cols):
    """Returns (lag_ms, median_pr, chosen_eeg_channel)."""
    if len(emg) < 10 or len(eeg) < 10 or (not emg_cols) or (not eeg_cols):
        return 0.0, np.nan, None

    tE = emg["Timestamp_seconds"].to_numpy(float)
    fsE = _infer_fs(tE)

    # Pick most energetic EMG raw channel (envelope)
    best_env, best_var, best_emg_ch = None, -np.inf, None
    for c in emg_cols:
        x = pd.to_numeric(emg[c], errors="coerce").to_numpy(float)
        x = _butter_lowpass(np.abs(np.nan_to_num(x)), fsE, EMG_ENV_LP)
        v = np.nanvar(x[np.isfinite(x)])
        if v > best_var:
            best_var, best_env, best_emg_ch = v, x, c

    tG = eeg["Timestamp_seconds"].to_numpy(float)
    fsG = _infer_fs(tG)

    ov_a, ov_b = _overlap_span(tE, tG)
    wins = _pick_windows_by_energy(tE, best_env, ov_a, ov_b)

    # Choose EEG channel by best median PR across windows
    pick = {"score": -np.inf, "ch": None}
    for ec in eeg_cols:
        gx = pd.to_numeric(eeg[ec], errors="coerce").to_numpy(float)
        gx = _butter_bandpass(np.nan_to_num(gx), fsG, EEG_BAND[0], EEG_BAND[1])

        prs = []
        for a, b in wins:
            _, env_h = _prep_corr_window(tE, best_env, XCORR_HZ, a, b, None)
            _, eeg_h = _prep_corr_window(tG, gx,      XCORR_HZ, a, b, None)
            if env_h is None or eeg_h is None:
                continue
            _, pr = lag_xcorr_ms(env_h, eeg_h, XCORR_HZ, EMG_EEG_SEARCH_MAX_MS)
            if np.isfinite(pr):
                prs.append(pr)

        if prs:
            score = float(np.median(prs))
            if score > pick["score"]:
                pick = {"score": score, "ch": ec}

    chosen = pick["ch"]
    if chosen is None:
        # small fallback based on median timestamps
        use = float(np.clip((_safe_float(np.nanmedian(tE)) - _safe_float(np.nanmedian(tG))) * 1000.0,
                             -FALLBACK_CLIP_MS, FALLBACK_CLIP_MS))
        return _grid_round_ms(use), np.nan, None

    # Estimate lag using both xcorr + gccphat across windows
    gx = pd.to_numeric(eeg[chosen], errors="coerce").to_numpy(float)
    gx = _butter_bandpass(np.nan_to_num(gx), fsG, EEG_BAND[0], EEG_BAND[1])

    lags, weights = [], []
    edge_hits = 0

    for a, b in wins[:N_SELECT]:
        _, env_h = _prep_corr_window(tE, best_env, XCORR_HZ, a, b, None)
        _, eeg_h = _prep_corr_window(tG, gx,      XCORR_HZ, a, b, None)
        if env_h is None or eeg_h is None:
            continue

        l1, pr1 = lag_xcorr_ms(env_h, eeg_h, XCORR_HZ, EMG_EEG_SEARCH_MAX_MS)
        l2, pr2 = lag_gccphat_ms(env_h, eeg_h, XCORR_HZ, EMG_EEG_SEARCH_MAX_MS)

        for lag, pr in ((l1, pr1), (l2, pr2)):
            if not (np.isfinite(lag) and np.isfinite(pr)):
                continue
            lags.append(lag)
            weights.append(max(pr, 1.0))
            if abs(lag) > EDGE_HIT_FRAC * EMG_EEG_SEARCH_MAX_MS:
                edge_hits += 1

    if len(lags) < MIN_ACCEPT:
        use = float(np.clip((_safe_float(np.nanmedian(tE)) - _safe_float(np.nanmedian(tG))) * 1000.0,
                             -FALLBACK_CLIP_MS, FALLBACK_CLIP_MS))
        return _grid_round_ms(use), np.nan, chosen

    # Agreement filter around median
    l = np.array(lags, float)
    w = np.array(weights, float)
    med = np.median(l)
    ok = np.abs(l - med) <= AGREE_TOL_MS
    if ok.any():
        l, w = l[ok], w[ok]

    # Add prior
    if EMG_EEG_PRIOR_MS is not None and np.isfinite(EMG_EEG_PRIOR_MS):
        l = np.append(l, EMG_EEG_PRIOR_MS)
        w = np.append(w, PRIOR_STRENGTH * (np.max(w) if len(w) else 1.0))

    # Weighted median
    order = np.argsort(l)
    l = l[order]
    w = w[order]
    cw = np.cumsum(w) / np.sum(w)
    use = float(l[min(np.searchsorted(cw, 0.5), len(l) - 1)])

    # Final guardrails
    if abs(use) > EMG_EEG_ABS_REJECT_MS or edge_hits >= 2:
        use = 0.0

    return _grid_round_ms(use), float(np.median(w)) if len(w) else np.nan, chosen


def _et_velocity_signal(et: pd.DataFrame):
    # Prefer X channels (your schema uses gaze normalized 0..1)
    candidates = ["ET_GazeLeftx", "ET_GazeRightx", "ET_GazeDirectionX", "ET_GazeX"]
    for name in candidates:
        if name in et.columns:
            x = pd.to_numeric(et[name], errors="coerce").to_numpy(float)
            t = et["Timestamp_seconds"].to_numpy(float)
            if len(x) >= 3:
                dt = np.diff(t)
                dt[dt <= 0] = np.nan
                v = np.empty_like(x, dtype=float)
                v[:] = np.nan
                v[1:] = np.diff(x) / dt
                return t, v
    return None, None


def estimate_lag_eeg_to_eye(eeg: pd.DataFrame, et: pd.DataFrame, eeg_ch: str | None):
    """Return EEG↔ET lag in ms (conservative)."""
    if eeg_ch is None or len(eeg) < 10 or len(et) < 10:
        return 0.0

    tG = eeg["Timestamp_seconds"].to_numpy(float)
    fsG = _infer_fs(tG)
    gx = pd.to_numeric(eeg[eeg_ch], errors="coerce").to_numpy(float)
    gx = _butter_bandpass(np.nan_to_num(gx), fsG, EEG_BAND[0], EEG_BAND[1])

    tT, vT = _et_velocity_signal(et)
    if tT is None:
        return 0.0

    fsT = _infer_fs(tT)
    vT = _butter_bandpass(np.nan_to_num(vT), fsT, ET_VEL_BAND[0], ET_VEL_BAND[1])

    ov_a, ov_b = _overlap_span(tT, tG)
    a, b = _pick_windows_by_energy(tT, vT, ov_a, ov_b, n_bins=3, n_select=1)[0]

    _, v_h = _prep_corr_window(tT, vT, XCORR_HZ, a, b, None)
    _, g_h = _prep_corr_window(tG, gx, XCORR_HZ, a, b, None)
    if v_h is None or g_h is None:
        return 0.0

    lag_ms, pr = lag_xcorr_ms(v_h, g_h, XCORR_HZ, ET_MAX_ABS_MS)
    if (not np.isfinite(lag_ms)) or (not np.isfinite(pr)) or pr < ET_PEAKR_MIN or abs(lag_ms) > ET_MAX_ABS_MS:
        return 0.0

    # Conservative revert check: require corr@lag > corr@0 + margin
    th = np.linspace(a, b, int((b-a)*XCORR_HZ), endpoint=False)
    v0 = _interp_series(tT, vT, th)
    g0 = _interp_series(tG, gx, th)
    if FILL_NAN_FOR_CORR:
        v0 = _nan_linear_fill(v0)
        g0 = _nan_linear_fill(g0)

    def _corr(u, v):
        u = (u - np.mean(u)) / (np.std(u) + 1e-12)
        v = (v - np.mean(v)) / (np.std(v) + 1e-12)
        return float(np.nan_to_num(np.corrcoef(u, v)[0, 1]))

    c0 = _corr(v0, g0)

    th_shift = th - (lag_ms / 1000.0)
    gL = _interp_series(tG, gx, th_shift)
    if FILL_NAN_FOR_CORR:
        gL = _nan_linear_fill(gL)

    cL = _corr(v0, gL)
    if (not np.isfinite(c0)) or (not np.isfinite(cL)) or (cL <= c0 + ET_REVERT_MARGIN):
        return 0.0

    return _grid_round_ms(lag_ms)


# =========================
# Grid + resampling
# =========================
def build_common_time_grid(emg, eeg, et):
    spans = []
    for seg in (emg, eeg, et):
        if len(seg):
            spans.append((
                float(np.nanmin(seg["Timestamp_seconds"])),
                float(np.nanmax(seg["Timestamp_seconds"]))
            ))
    if not spans:
        return np.array([])

    if SPAN_STYLE == "overlap":
        t0 = max(s for s, _ in spans)
        t1 = min(e for _, e in spans)
    else:
        t0 = min(s for s, _ in spans)
        t1 = max(e for _, e in spans)

    if not np.isfinite(t0) or not np.isfinite(t1) or t1 <= t0:
        return np.array([])

    if GRID_STYLE == "fixed":
        n = int(np.floor((t1 - t0) * TARGET_HZ))
        return t0 + np.arange(n) / TARGET_HZ

    # Native: prefer EMG then EEG then ET
    if len(emg): return emg["Timestamp_seconds"].to_numpy(float)
    if len(eeg): return eeg["Timestamp_seconds"].to_numpy(float)
    return et["Timestamp_seconds"].to_numpy(float)


def resample_stream(seg, cols, t_grid, lag_ms=0.0):
    """
    Shift by lag (source time -> align to reference), then interpolate onto grid.
    Returns (dict_of_arrays, list_of_aa_columns)
    """
    if len(seg) == 0 or len(t_grid) == 0 or not cols:
        return {}, []

    t0 = seg["Timestamp_seconds"].to_numpy(float)
    fs_native = _infer_fs(t0)
    t_src = t0 - (lag_ms / 1000.0)

    out = {}
    aa_cols = []
    for c in cols:
        if c not in seg.columns:
            continue
        y = pd.to_numeric(seg[c], errors="coerce").to_numpy(float)

        did_aa = False
        if (AA_ON and GRID_STYLE == "fixed"
            and np.isfinite(fs_native) and fs_native > TARGET_HZ * AA_TOL
            and c not in DISCRETE_COLS):
            y = _aa_pre_filter(y, fs_native, TARGET_HZ)
            did_aa = True

        kind = "nearest" if c in DISCRETE_COLS else "linear"
        out[c] = _interp_series(t_src, y, t_grid, kind=kind)

        if did_aa:
            aa_cols.append(c)

    return out, aa_cols


# =========================
# File runner
# =========================
def sync_one_csv(csv_path: str | Path, out_dir: str | Path) -> dict:
    csv_path = str(csv_path)
    fname = os.path.basename(csv_path)
    stem = Path(fname).stem

    print("\n" + "="*72)
    print(f"Syncing: {fname}")
    print("="*72)

    df = ingest_clean_csv(csv_path)
    meta = extract_constant_meta(df) if KEEP_META_COLUMNS else {}

    emg, eeg, et, emg_cols, eeg_cols, et_cols = partition_modal_streams(df)

    print(f"Rows (usable) → EMG:{len(emg)} EEG:{len(eeg)} ET:{len(et)}")
    fs_emg = _infer_fs(emg["Timestamp_seconds"].to_numpy(float)) if len(emg) else 0.0
    fs_eeg = _infer_fs(eeg["Timestamp_seconds"].to_numpy(float)) if len(eeg) else 0.0
    fs_et  = _infer_fs(et["Timestamp_seconds"].to_numpy(float))  if len(et)  else 0.0
    print(f"Native fs  → EMG={fs_emg:.1f} Hz  EEG={fs_eeg:.1f} Hz  ET={fs_et:.1f} Hz")

    lag_emg_eeg_ms, pr_med, eeg_star = estimate_lag_emg_to_eeg(emg, eeg, emg_cols, eeg_cols)
    print(f"EMG↔EEG lag: {lag_emg_eeg_ms:+.1f} ms | EEG*={eeg_star} | PR~{pr_med if np.isfinite(pr_med) else np.nan}")

    lag_eeg_et_ms = estimate_lag_eeg_to_eye(eeg, et, eeg_star)
    print(f"EEG↔ET  lag: {lag_eeg_et_ms:+.1f} ms  ({'xcorr' if lag_eeg_et_ms != 0 else 'safe=0'})")

    # Derived for reference
    lag_emg_et_ms = float(lag_emg_eeg_ms - lag_eeg_et_ms)

    t_grid = build_common_time_grid(emg, eeg, et)
    if len(t_grid) == 0:
        raise RuntimeError("Empty time grid (check timestamps / span mode).")

    print(f"Grid: {GRID_STYLE} @ {TARGET_HZ} Hz | rows={len(t_grid)} | SPAN={SPAN_STYLE}")

    out = {}
    # meta constants (broadcast)
    for k, v in meta.items():
        out[k] = np.full(len(t_grid), v, dtype=float)

    out["Timestamp_seconds"] = t_grid
    out["Timestamp_ms"] = t_grid * 1000.0

    aa_log = {"emg": [], "eeg": [], "et": []}

    if emg_cols:
        emg_map, aa_emg = resample_stream(emg, emg_cols, t_grid, lag_ms=lag_emg_eeg_ms)
        out.update(emg_map)
        aa_log["emg"] = aa_emg
        print(f"Added EMG: {len(emg_map)} cols | AA:{len(aa_emg)}")

    if eeg_cols:
        eeg_map, aa_eeg = resample_stream(eeg, eeg_cols, t_grid, lag_ms=0.0)
        out.update(eeg_map)
        aa_log["eeg"] = aa_eeg
        print(f"Added EEG: {len(eeg_map)} cols | AA:{len(aa_eeg)}")

    if et_cols:
        et_map, aa_et = resample_stream(et, et_cols, t_grid, lag_ms=lag_eeg_et_ms)
        out.update(et_map)
        aa_log["et"] = aa_et
        print(f"Added ET : {len(et_map)} cols | AA:{len(aa_et)}")

    synced = pd.DataFrame(out).dropna(axis=1, how="all")

    # Round numeric floats for compactness
    for c in synced.select_dtypes(include=["float64", "float32"]).columns:
        synced[c] = synced[c].round(ROUND_DECIMALS)

    os.makedirs(out_dir, exist_ok=True)
    out_csv = os.path.join(out_dir, f"{stem}_synchronized_corrected.csv")
    synced.to_csv(out_csv, index=False, float_format=f"%.{ROUND_DECIMALS}f")

    try:
        sz = os.path.getsize(out_csv) / (1024 * 1024)
        print(f"Saved CSV: {out_csv} ({sz:.1f} MB)")
    except Exception:
        print(f"Saved CSV: {out_csv}")

    if WRITE_JSON:
        log = {
            "input_file": str(Path(csv_path).resolve()),
            "output_csv": str(Path(out_csv).resolve()),
            "grid_style": GRID_STYLE,
            "grid_hz": TARGET_HZ,
            "span_style": SPAN_STYLE,
            "lag_emg_eeg_ms": float(lag_emg_eeg_ms),
            "lag_eeg_et_ms": float(lag_eeg_et_ms),
            "lag_emg_et_ms": float(lag_emg_et_ms),
            "chosen_eeg_channel": eeg_star if eeg_star else "",
            "median_pr_emg_eeg": None if (pr_med is None or not np.isfinite(pr_med)) else float(pr_med),
            "aa_applied_columns": aa_log,
            "meta_constants": meta,
            "notes": "Lags are applied BEFORE interpolation. Positive lag means source lags reference; alignment uses t_src = t - lag."
        }
        out_json = os.path.join(out_dir, f"{stem}_synchronized_corrected.json")
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(log, f, indent=2)
        print(f"Saved JSON: {out_json}")

    return {
        "filename": fname,
        "rows_emg": len(emg), "rows_eeg": len(eeg), "rows_et": len(et),
        "grid_rows": len(t_grid),
        "fs_emg": fs_emg, "fs_eeg": fs_eeg, "fs_et": fs_et,
        "lag_emg_eeg_ms": float(lag_emg_eeg_ms),
        "lag_eeg_et_ms": float(lag_eeg_et_ms),
        "lag_emg_et_ms": float(lag_emg_et_ms),
        "best_eeg_ch": eeg_star,
        "aa_emg": len(aa_log["emg"]),
        "aa_eeg": len(aa_log["eeg"]),
        "aa_et": len(aa_log["et"]),
        "meta": meta
    }


def run_sync_batch():
    print("Starting tri-modal synchronization (UnionGrid Lite v3)")
    print(f"GRID_STYLE={GRID_STYLE} | TARGET_HZ={TARGET_HZ} | SPAN_STYLE={SPAN_STYLE}")
    print("IN :", IN_DIR)
    print("OUT:", OUT_DIR)

    os.makedirs(OUT_DIR, exist_ok=True)

    files = sorted(glob.glob(os.path.join(IN_DIR, "*.csv")))
    files = [f for f in files if "_synchronized" not in os.path.basename(f).lower()]
    print(f"Found {len(files)} files")

    summary = []
    for i, fp in enumerate(files, 1):
        print(f"\n[{i}/{len(files)}]")
        try:
            summary.append(sync_one_csv(fp, OUT_DIR))
        except Exception as e:
            print(f"❌ Error on {os.path.basename(fp)}: {e}")
            summary.append({"filename": os.path.basename(fp), "status": f"ERROR: {e}"})

    if summary:
        out_sum = os.path.join(OUT_DIR, "synchronization_summary.csv")
        pd.DataFrame(summary).to_csv(out_sum, index=False)
        print(f"\nSummary saved: {out_sum}")

    print("\n" + "="*80)
    print("SYNC COMPLETE")
    print("="*80)


# if __name__ == "__main__":
#     run_sync_batch()


In [4]:
from pathlib import Path
import re

ROOT = Path("/home/tsultan1/IROS/Data")

def sort_key_sub(p: Path):
    m = re.search(r"Sub-(\d+)$", p.name)
    return int(m.group(1)) if m else 10**9

subject_dirs = sorted([p for p in ROOT.glob("Sub-*") if p.is_dir()], key=sort_key_sub)

for sub in subject_dirs:
    IN_DIR  = str(sub / "cleaned")
    OUT_DIR = str(sub / "cleaned" / "synchronized_proper_lite_union_v3")

    if not Path(IN_DIR).exists():
        print(f"[SKIP] {sub.name}: missing cleaned folder")
        continue

    print(f"\n=== {sub.name} ===")
    run_sync_batch()



=== Sub-1 ===
Starting tri-modal synchronization (UnionGrid Lite v3)
GRID_STYLE=fixed | TARGET_HZ=250 | SPAN_STYLE=all
IN : /home/tsultan1/IROS/Data/Sub-1/cleaned
OUT: /home/tsultan1/IROS/Data/Sub-1/cleaned/synchronized_proper_lite_union_v3
Found 83 files

[1/83]

Syncing: 001_T03.csv
Rows (usable) → EMG:6118 EEG:8824 ET:16789
Native fs  → EMG=500.0 Hz  EEG=500.9 Hz  ET=1190.3 Hz
EMG↔EEG lag: -20.0 ms | EEG*=Ch2 | PR~4.4974547635677276
EEG↔ET  lag: +0.0 ms  (safe=0)
Grid: fixed @ 250 Hz | rows=3060 | SPAN=all
Added EMG: 4 cols | AA:4
Added EEG: 8 cols | AA:8
Added ET : 22 cols | AA:17
Saved CSV: /home/tsultan1/IROS/Data/Sub-1/cleaned/synchronized_proper_lite_union_v3/001_T03_synchronized_corrected.csv (0.9 MB)
Saved JSON: /home/tsultan1/IROS/Data/Sub-1/cleaned/synchronized_proper_lite_union_v3/001_T03_synchronized_corrected.json

[2/83]

Syncing: 002_T516.csv
Rows (usable) → EMG:5164 EEG:7444 ET:14137
Native fs  → EMG=500.0 Hz  EEG=500.6 Hz  ET=1021.8 Hz
EMG↔EEG lag: -28.0 ms | EEG*=C